# Assignment 1 - Step 1

In [93]:
#Import relevant libraries
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [94]:
# import wind CF data for 6 windfarms and cut to be 24 hours and average out

W1_data = pd.read_csv(r'data_from_Jakob\scen_zone1.csv')
W1_CF = W1_data.iloc[1:25, 1:].mean(axis=1)

W2_data = pd.read_csv(r'data_from_Jakob\scen_zone2.csv')
W2_CF = W2_data.iloc[1:25, 1:].mean(axis=1)

W3_data = pd.read_csv(r'data_from_Jakob\scen_zone3.csv')
W3_CF = W3_data.iloc[1:25, 1:].mean(axis=1)

W4_data = pd.read_csv(r'data_from_Jakob\scen_zone4.csv')
W4_CF = W4_data.iloc[1:25, 1:].mean(axis=1)

W5_data = pd.read_csv(r'data_from_Jakob\scen_zone5.csv')
W5_CF = W5_data.iloc[1:25, 1:].mean(axis=1)

W6_data = pd.read_csv(r'data_from_Jakob\scen_zone6.csv')
W6_CF = W6_data.iloc[1:25, 1:].mean(axis=1)


In [95]:
from data import load_distribution, load_profile, generators, generator_bid_prices
from data import Prices_for_loads, transmission_lines, Zonal_networks

#create variable list with the names of the variables as strings
VARIABLES = list(generators.keys())

LOAD_VARIABLES = list(load_distribution.keys())

#VOLTAGE_ANGLES = [f"theta_{i}" for i in range(1, 25)]

POWER_FLOW_VARIABLES = ["P_flow_Z1toZ2", "P_flow_Z1toZ3", "P_flow_Z2toZ3"]

#

#create a list of the cost coefficients for each variable
Generation_price = [v[12] for v in generator_bid_prices.values()]

#writes a list that converts the cost coefficient to this form objective_coeff = {'G1': 13.32, 'G2': 13.32, ...}
objective_coeff = {VARIABLES[i]: Generation_price[i] for i in range(len(VARIABLES))}



#write the load percentages for each node
Load_percentage = [v['percent'] for v in load_distribution.values()]
#multiply the load percentage with the total load to get the actual load in MW
hour12_load = load_profile[12] #MW

#Load upper bound list for each load variable
Load = [hour12_load * (i / 100) for i in Load_percentage] #MW


#random price list with 17 values and sort it in descending order
#change prices for clarity in the plot
Randonm_prices_h1 = Prices_for_loads[12]
Demand_price= np.array(sorted(Randonm_prices_h1, reverse=True)) -60 #$/MWh 

#writes a list that converts the load coefficient to this form objective_coeff = {'Load1': 13.32, 'Load2': 13.32, ...}
Load_coefficients = {LOAD_VARIABLES[i]: Demand_price[i] for i in range(len(LOAD_VARIABLES))}

#create a upper bound for the production variables
Generator_UB = [v['Pmax_MW'] for k, v in generators.items() if k.startswith('G')]

#Generator_UB = [152,152,350,591,60,155,155,400,400,300,310,350] #MW
Wind_UB = np.array([W1_CF[12], W2_CF[12], W3_CF[12], W4_CF[12],W5_CF[12],W6_CF[12]])*200
Constraints_rhs = Generator_UB + Wind_UB.tolist()
#create constraints sense list with 18 values of GRB.LESS_EQUAL
constraints_sense = [GRB.LESS_EQUAL] * 18




In [96]:
# defining the ATC for the inter-zonal lines
interzonal_lines = {
    ("Z1", "Z2"): [],
    ("Z1", "Z3"): [],
    ("Z2", "Z3"): []
}

for l, data in transmission_lines.items():
    f_node = data['from']
    t_node = data['to']

    # Find zones
    f_zone = [z for z, val in Zonal_networks.items() if f_node in val['nodes']][0]
    t_zone = [z for z, val in Zonal_networks.items() if t_node in val['nodes']][0]

    if f_zone != t_zone:
        # store line key in the correct tuple (alphabetical order to avoid duplicates)
        key = tuple(sorted([f_zone, t_zone]))
        interzonal_lines[key].append(l)

ATC = {}
for zones, lines in interzonal_lines.items():
    ATC[zones[0] + "to" + zones[1]] = sum(transmission_lines[l]['capacity_MVA'] for l in lines)




In [97]:
#Create model
model = gp.Model("Network_Zonal_singlehour")

In [98]:
#Add variables
variables = {v: model.addVar(lb=0, name=f'variable {v}') for v in VARIABLES}

#Add load variables
load_variables = {l: model.addVar(lb=0, name=f'variable {l}') for l in LOAD_VARIABLES}

#Add power flow variables
flow_variables = {f: model.addVar(lb=-GRB.INFINITY, name=f'variable {f}') for f in POWER_FLOW_VARIABLES}



In [99]:
# Set objective function and optimization direction of the Gurobi model
objective = gp.quicksum(Load_coefficients[v] * load_variables[v] for v in LOAD_VARIABLES) - gp.quicksum(objective_coeff[v] * variables[v] for v in VARIABLES)
model.setObjective(objective, GRB.MAXIMIZE)


In [100]:
# Add constraints to the Gurobi model

#Add balance constraint to the Gurobi model that ensures that the total load is equal to the total generation
Balance_constraints = {}

zones = list(Zonal_networks.keys())

for z in zones:

    nodes_in_zone = Zonal_networks[z]['nodes']

    generation = gp.quicksum(
        variables[g]
        for g in variables
        if generators[g]['node'] in nodes_in_zone
    )

    load = gp.quicksum(
        load_variables[l]
        for l in load_variables
        if load_distribution[l]['node'] in nodes_in_zone
    )

    flow = 0

    if z == "Z1":
        flow = flow_variables["P_flow_Z1toZ2"] + flow_variables["P_flow_Z1toZ3"]

    if z == "Z2":
        flow = -flow_variables["P_flow_Z1toZ2"] + flow_variables["P_flow_Z2toZ3"]

    if z == "Z3":
        flow = -flow_variables["P_flow_Z1toZ3"] - flow_variables["P_flow_Z2toZ3"]

    Balance_constraints[z] = model.addLConstr(
        - generation + load + flow,
        GRB.EQUAL,
        0,
        name=f"Balance_constraint_{z}"
    )



#Add boundary constraints to the Gurobi model that ensures that the generation of each generator is less than or equal to its upper bound
Boundary_constraints = {}

for v, rhs in zip(VARIABLES, Constraints_rhs):
    Boundary_constraints[v] = model.addLConstr(
        variables[v],
        GRB.LESS_EQUAL,
        rhs
    )

Boundary_constraints2 = {}
for v, rhs in zip(LOAD_VARIABLES, Load):
    Boundary_constraints2[v] = model.addLConstr(
        load_variables[v],
        GRB.LESS_EQUAL,
        rhs
    )

ATC_constraints = {}

for link, cap in ATC.items():
    # Upper bound
    ATC_constraints[f"{link}_upper"] = model.addLConstr(
        flow_variables[f"P_flow_{link}"], GRB.LESS_EQUAL, cap
    )
    # Lower bound
    ATC_constraints[f"{link}_lower"] = model.addLConstr(
        flow_variables[f"P_flow_{link}"], GRB.GREATER_EQUAL, -cap
    )

#Add ATC constraints for the inter-zonal lines

In [101]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 44 rows, 38 columns and 82 nonzeros (Max)
Model fingerprint: 0x3585bc09
Model has 28 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 7e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e+01, 2e+03]

Presolve removed 42 rows and 18 columns
Presolve time: 0.02s
Presolved: 2 rows, 20 columns, 21 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.1208229e+05   2.623445e+02   0.000000e+00      0s
       2    1.0122806e+05   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.03 seconds (0.00 work units)
Optimal objective  1.012280640e+05


In [102]:
# Check that balance is maintained at each zone
for z in zones:
    nodes_in_zone = Zonal_networks[z]['nodes']

    # Sum generation in this zone
    generation = sum(variables[v].X for v in variables if generators[v]['node'] in nodes_in_zone)

    # Sum load in this zone
    load = sum(load_variables[v].X for v in load_variables if load_distribution[v]['node'] in nodes_in_zone)

    # Compute net flow leaving the zone
    if z == "Z1":
        net_flow = flow_variables["P_flow_Z1toZ2"].X + flow_variables["P_flow_Z1toZ3"].X
    elif z == "Z2":
        net_flow = -flow_variables["P_flow_Z1toZ2"].X + flow_variables["P_flow_Z2toZ3"].X
    elif z == "Z3":
        net_flow = -flow_variables["P_flow_Z1toZ3"].X - flow_variables["P_flow_Z2toZ3"].X

    # Net balance should be close to zero
    net_balance = generation - load - net_flow
    print(f"sum of balance in zone {z}: {net_balance:.6f} MW (should be close to 0)")

sum of balance in zone Z1: -0.000000 MW (should be close to 0)
sum of balance in zone Z2: 0.000000 MW (should be close to 0)
sum of balance in zone Z3: 0.000000 MW (should be close to 0)


In [103]:
# write all the market prices
if model.status == GRB.OPTIMAL:
    for z in zones:
        print(f"Market price in zone {z}: {Balance_constraints[z].Pi:.2f} $/MWh")

Market price in zone Z1: 10.52 $/MWh
Market price in zone Z2: 10.52 $/MWh
Market price in zone Z3: 10.52 $/MWh


In [104]:
if model.status == GRB.OPTIMAL:
    print(f"Optimal objective: {model.ObjVal:.2f} EUR")
    for i in VARIABLES:
        print(f"Optimal dispatch of generator {i}: {variables[i].X:.2f} MW")
    for i in LOAD_VARIABLES:
        print(f"Optimal dispatch of load {i}: {load_variables[i].X:.2f} MW")

Optimal objective: 101228.06 EUR
Optimal dispatch of generator G1: 0.00 MW
Optimal dispatch of generator G2: 0.00 MW
Optimal dispatch of generator G3: 0.00 MW
Optimal dispatch of generator G4: 0.00 MW
Optimal dispatch of generator G5: 0.00 MW
Optimal dispatch of generator G6: 129.89 MW
Optimal dispatch of generator G7: 155.00 MW
Optimal dispatch of generator G8: 400.00 MW
Optimal dispatch of generator G9: 400.00 MW
Optimal dispatch of generator G10: 300.00 MW
Optimal dispatch of generator G11: 310.00 MW
Optimal dispatch of generator G12: 0.00 MW
Optimal dispatch of generator W1: 134.44 MW
Optimal dispatch of generator W2: 141.30 MW
Optimal dispatch of generator W3: 145.36 MW
Optimal dispatch of generator W4: 125.83 MW
Optimal dispatch of generator W5: 137.37 MW
Optimal dispatch of generator W6: 138.79 MW
Optimal dispatch of load Load1: 95.68 MW
Optimal dispatch of load Load2: 85.61 MW
Optimal dispatch of load Load3: 158.63 MW
Optimal dispatch of load Load4: 65.47 MW
Optimal dispatch of

In [110]:
# Example: total profits per zone

# Map generators and loads to zones
generator_zone = {g: next(z for z, val in Zonal_networks.items() if generators[g]['node'] in val['nodes'])
                  for g in generators}
zonal_gen_profit = {z: 0 for z in Zonal_networks}

# Production surplus and generator profits per zone
Production_surplus = 0
print("=== Generator profits ===")
for g in generators:
    z = generator_zone[g]
    price = Balance_constraints[z].Pi  # Zonal price
    profit = (price - generator_bid_prices[g][12]) * variables[g].X  # hour 12 example
    Production_surplus += profit
    print(f"Generator {g} (zone {z}) profit: {profit:.2f} EUR")

print(f"Total production surplus: {Production_surplus:.2f} EUR\n")
print(f"Total Social Welfare: {model.ObjVal:.2f} EUR")


=== Generator profits ===
Generator G1 (zone Z2) profit: -0.00 EUR
Generator G2 (zone Z2) profit: -0.00 EUR
Generator G3 (zone Z3) profit: -0.00 EUR
Generator G4 (zone Z3) profit: -0.00 EUR
Generator G5 (zone Z1) profit: -0.00 EUR
Generator G6 (zone Z1) profit: 0.00 EUR
Generator G7 (zone Z1) profit: 0.00 EUR
Generator G8 (zone Z1) profit: 1800.00 EUR
Generator G9 (zone Z1) profit: 2020.00 EUR
Generator G10 (zone Z1) profit: 3156.00 EUR
Generator G11 (zone Z1) profit: 0.00 EUR
Generator G12 (zone Z1) profit: -0.00 EUR
Generator W1 (zone Z2) profit: 1414.30 EUR
Generator W2 (zone Z2) profit: 1486.45 EUR
Generator W3 (zone Z3) profit: 1529.16 EUR
Generator W4 (zone Z1) profit: 1323.75 EUR
Generator W5 (zone Z1) profit: 1445.14 EUR
Generator W6 (zone Z1) profit: 1460.07 EUR
Total production surplus: 15634.87 EUR

Total Social Welfare: 101228.06 EUR


In [112]:
#check balance at each node

for n in range(1, 25):
    # total generation and load at node n
    generation = sum(variables[v].X for v in variables if generators[v]['node'] == n)
    load = sum(load_variables[v].X for v in load_variables if load_distribution[v]['node'] == n)
    
    net_injection = generation - load  # positive = surplus, negative = deficit
    
    # sum of capacities of lines connected to this node
    connected_lines = [l for l, data in transmission_lines.items() if data['from'] == n or data['to'] == n]
    total_capacity = sum(transmission_lines[l]['capacity_MVA'] for l in connected_lines)
    
    # print summary
    print(f"Node {n}: net injection = {net_injection:.2f} MW, total connected capacity = {total_capacity} MW")
    
    # rough feasibility check
    if abs(net_injection) > total_capacity:
        print(f"  --> Potential infeasibility! Ex-post re-dispatch might be needed: {abs(net_injection) - total_capacity:.2f} MW")



Node 1: net injection = -95.68 MW, total connected capacity = 700 MW
Node 2: net injection = -85.61 MW, total connected capacity = 525 MW
Node 3: net injection = -24.19 MW, total connected capacity = 750 MW
Node 4: net injection = -65.47 MW, total connected capacity = 350 MW
Node 5: net injection = 78.35 MW, total connected capacity = 700 MW
Node 6: net injection = -120.86 MW, total connected capacity = 350 MW
Node 7: net injection = 34.57 MW, total connected capacity = 350 MW
Node 8: net injection = -151.08 MW, total connected capacity = 700 MW
Node 9: net injection = -153.60 MW, total connected capacity = 1325 MW
Node 10: net injection = -171.22 MW, total connected capacity = 1500 MW
Node 11: net injection = 0.00 MW, total connected capacity = 1800 MW
Node 12: net injection = 0.00 MW, total connected capacity = 1800 MW
Node 13: net injection = -234.17 MW, total connected capacity = 1250 MW
Node 14: net injection = -171.22 MW, total connected capacity = 750 MW
Node 15: net injection =